# LB 23.91003: Exact Rewrites of the 892-Qubit DQI

**Leaderboard 23.91003** for this exact zip (version 14's output, scored 2 November). Local geometric mean at the
verifiers' default seeds: 23.91005.

**What is new in this version.** Two exact rewrites of SomeJoint's 892-qubit DQI decoder. First, 24 blocks that only
clear a flag by recomputing it are replaced by measurement-based uncomputation (12 of them are the sites of
manwithacat's HW-07, 12 more are three-input flags). Then, as in version 12, repeated AND products are cached on wires
that are provably idle and reused through CNOTs (two caching passes). Same qubits, same answers on every shot we simulated, **2,803 fewer Toffolis per shot** (399,969.7 against
402,772.3, -0.70%). The invsqrt and chemistry files are other authors'
public circuits, unchanged and credited at the end.

| task | circuit | qubits | cost (official verifier, default seed) | reference / cost |
|---|---|---:|---:|---:|
| invsqrt | Ehan Mallick's 62-qubit file, unchanged | 62 | 100,568.50 | **20.6503** |
| DQI | SomeJoint's 892-qubit decoder + our two exact rewrites | 892 | 3.791085e+11 | **64.4656** |
| chemistry | heliosli's weights on Lera's (9, 143, 40) table, unchanged | 428 | 326,801,702.15 | **10.2680** |

Where the DQI saving comes from (Toffolis per shot against SomeJoint's file):

| rewrite | saved per shot | how it was measured |
|---|---:|---|
| clearing blocks by measurement (24 sites) | about 206 | the builder's rate model |
| product caching, first pass (2,340 chains) | about 2,590 | the builder's rate model |
| product caching, second pass | 6.1 | official verifier, 3 paired seeds |
| **total** | **2,802.6** | official verifier, each file at its default seed |

Geometric mean at the verifiers' default seeds: **23.91005**. The leaderboard draws its own DQI samples, so its reading
can differ by a failed shot or two. The next cell checks the 18 files by SHA-256, runs the three official verifiers
unchanged (about 15 minutes on Kaggle's CPU) and writes `/kaggle/working/submission.zip`.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

COMP = next(p for p in (Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation'),
                        Path('/kaggle/input/google-verifiable-quantum-circuit-compilation')) if (p / 'verifiers').is_dir())
DS = next(p.parent for p in Path('/kaggle/input').rglob('SHA256SUMS') if (p.parent / 'dqi.kmb').is_file())
FILES = {
"dfthcsos__angle_k_0.kmb": [
"dfthcsos/angle_k_0.kmb",
"8b698f14b339bd282e0332a1f105b41d60bfa5218d107c775db2532fbd88f774"
],
"dfthcsos__angle_k_1.kmb": [
"dfthcsos/angle_k_1.kmb",
"0d16c5108e48e40931d87f1e3ee12d4486e932b341989ce909a47b032e5a7745"
],
"dfthcsos__angle_k_10.kmb": [
"dfthcsos/angle_k_10.kmb",
"e2668bf5666ca1e49541bef7a255b2100a8a513188fadef3a8dd4e892cfe2094"
],
"dfthcsos__angle_k_11.kmb": [
"dfthcsos/angle_k_11.kmb",
"5f69f385cb2040e7124e11e1cedb7cea0e0ffa6ad13abe81b2a337744cb9c2ac"
],
"dfthcsos__angle_k_12.kmb": [
"dfthcsos/angle_k_12.kmb",
"f86995a0ea1990a11a43f1812f47e00a77658f71cc544d44bd5003e616111272"
],
"dfthcsos__angle_k_13.kmb": [
"dfthcsos/angle_k_13.kmb",
"4390c8114e191b0c74ccd8b6302e5e0ded4e8cb412a3f146eea9887551658876"
],
"dfthcsos__angle_k_14.kmb": [
"dfthcsos/angle_k_14.kmb",
"9e596deb120f20d1c92c350456ce9847314ba943a4d196e7942bac518a229f6c"
],
"dfthcsos__angle_k_2.kmb": [
"dfthcsos/angle_k_2.kmb",
"b7b73491d3aeb59c70d023704df9f2ad69eb96c9d1f9e9920d33d60bd7852d11"
],
"dfthcsos__angle_k_3.kmb": [
"dfthcsos/angle_k_3.kmb",
"c2437ad406d3772e961e7229580fba7b68c036b97ce36b4067c865e3af729f00"
],
"dfthcsos__angle_k_4.kmb": [
"dfthcsos/angle_k_4.kmb",
"0df35ae1cb26d3bc09f022b2a397a8b07e5c322283d2c8cf0a53a4bb11b9dd47"
],
"dfthcsos__angle_k_5.kmb": [
"dfthcsos/angle_k_5.kmb",
"a9a649997bf0696a659c8cbf9e778f92bee2816b6212febc898ed51450ae42c3"
],
"dfthcsos__angle_k_6.kmb": [
"dfthcsos/angle_k_6.kmb",
"715673382c491b796865dbba235687eafebd16c745872d20f7ae1808f88a62ae"
],
"dfthcsos__angle_k_7.kmb": [
"dfthcsos/angle_k_7.kmb",
"31e004f932d42750e1dda98abdf3e0097941cb1b1bbd353ed692e52098119cfb"
],
"dfthcsos__angle_k_8.kmb": [
"dfthcsos/angle_k_8.kmb",
"046ac97b8c6ac9e1895a1da5dc89ea5df55bf487b1822d2ba11697c948145ff9"
],
"dfthcsos__angle_k_9.kmb": [
"dfthcsos/angle_k_9.kmb",
"10c37a55ab902357cf60c5cf529de0eec9e9497a03f0b2ad7a9888d26d9c1e63"
],
"dfthcsos__submission.npz": [
"dfthcsos/submission.npz",
"335c08fcb6c121fdd767516fd637b6a8ba31287723c88221ef54cabe9357e432"
],
"dqi.kmb": [
"dqi.kmb",
"92750bd64525924ef84c8cd1b379b2c2d3914d87347bbe1bc40c804efff3ccdc"
],
"invsqrt.kmb": [
"invsqrt.kmb",
"bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47"
]
}
EXPECTED = {"invsqrt": 100568.4991945597, "dqi": 379108516959.91815, "dfthcsos": 326801702.1535616}
REF = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "dfthcsos": 3355599588.3155365}
SUB = Path('/kaggle/working/sub')
shutil.rmtree(SUB, ignore_errors=True)
for name, (path, sha) in FILES.items():
    dst = SUB / path
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DS / name, dst)
    assert hashlib.sha256(dst.read_bytes()).hexdigest() == sha, name
print(len(FILES), 'files copied and matched by SHA-256 from', DS)

In [ ]:
ENV = dict(os.environ, PYTHONPATH=f"{COMP / 'kickmix'}:{COMP / 'verifiers'}")
RUNS = {'invsqrt': ['invsqrt/verify_submission_invsqrt.py', str(SUB / 'invsqrt.kmb')],
        'dqi': ['dqi/verify_submission_dqi.py', str(SUB / 'dqi.kmb')],
        'dfthcsos': ['dfthcsos/verify.py', str(SUB / 'dfthcsos')]}
res = {}
for task, args in RUNS.items():
    t0 = time.time()
    r = subprocess.run([sys.executable] + args, cwd=COMP / 'verifiers', env=ENV, capture_output=True, text=True)
    out = r.stdout
    res[task] = json.loads(out[out.index('{'):])
    print(f"{task:9s} {res[task]['status']}  cost {res[task]['score']:.10g}  ({time.time() - t0:.0f} s)", flush=True)
for task, r in res.items():
    assert r['status'] == 'PASS', (task, r.get('error'))
    assert abs(r['score'] / EXPECTED[task] - 1) < 1e-9, (task, r['score'], EXPECTED[task])
ratios = {t: REF[t] / res[t]['score'] for t in REF}
print({t: round(v, 4) for t, v in ratios.items()})
print(f"geometric mean of reference / cost: {(ratios['invsqrt'] * ratios['dqi'] * ratios['dfthcsos']) ** (1 / 3):.5f}")

In [ ]:
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name, (path, sha) in FILES.items():
        z.write(SUB / path, path)
with zipfile.ZipFile('/kaggle/working/submission.zip') as z:
    assert z.testzip() is None and sorted(z.namelist()) == sorted(p for p, _ in FILES.values())
    for name, (path, sha) in FILES.items():
        assert hashlib.sha256(z.read(path)).hexdigest() == sha, path
shutil.rmtree(SUB)
print('wrote /kaggle/working/submission.zip', os.path.getsize('/kaggle/working/submission.zip'), 'bytes')

## How the caching works, and how we checked it

**Finding repeats.** A symbolic pass walks the circuit and gives every wire, at every gate, a hash of the Boolean
function it holds: inputs, measurement outcomes and the classical masks are fresh variables, CNOTs XOR the hashes and
an AND makes a new product. Two Toffoli-like gates whose control pair holds the same product (same hash) compute the
same AND on every input. On this decoder 6,860 such candidates group into chains of repeated products.

**Placing a copy.** A chain becomes: one Toffoli writes the product onto a wire that is provably zero there and touched
by no gate until the clear; each later use becomes a CNOT (CCX), a CZ (CCZ) or three CNOTs (CSWAP) from that copy; the
clear measures the copy in the X basis and, if the outcome is 1, applies a CZ on the operand pair (measurement-based
uncomputation, a trick due to Craig Gidney). The operands of the first and last use must be untouched between the copy
and its uses. Free wires are the binding limit at 892 qubits, so an exact linear program picks which chains to place.
Free wires, not repeats, are the limit.

**Clearing blocks.** In the reverse of extraction (`unextract`), each of the 12 epochs clears two flags by recomputing
them: a 16-Toffoli ladder for a six-input AND and a 4-Toffoli ladder for a three-input AND. We measure each flag in the X
basis instead; only on outcome 1 a phase fix is due, a CCZ for the three-input flag and manwithacat's two-wire clean
ladder for the six-input one (HW-07, about 2.25 Toffolis expected). Each of the 24 sites was proven exact first (by the
symbolic pass or exhaustively over all 4,096 values of its free inputs): 240 static Toffolis removed. Caching then
runs on the rewritten circuit: 2,340 chains (1,420 CCX, 3,491 CSWAP and 216 CCZ uses), 2,787 static Toffolis removed.

**Checks.** Official verifier, unchanged, at three seeds (the member's default seed and two fresh salts): 171.0 fewer
Toffolis per shot than our version-12 file at each seed, errors equal. An independent bit-parallel simulator compared it
with SomeJoint's file on 32,768 shots, including shots where every phase fix is forced to run: not one differs. The
second caching pass saves 6.1 more per shot at three seeds with equal errors, and the same simulator finds 0 differing
shots of 24,576 between it and the one-pass file. On the
first file we built this way (on yoyobar's decoder, -0.64%), an independent review rebuilt it byte for byte,
reran the verifier at the maximum error weight (35) and ran its own bit-parallel simulator with forced and biased
measurement outcomes: 161,792 shots, not one differing from the original circuit in any qubit, bit or phase. It found
three holes in the method that no file had hit (a hash collision in the fresh variables, conditional Toffolis, copies
moved into conditional blocks); all three are fixed and tested before this file was built.

**What it is not.** It does not change the decoder's algorithm. Adding a qubit to cache more would buy about 263
Toffolis per shot against about 900 that a qubit costs at this width, so 892 stays.

## Credits and licence

| author | what | source | licence |
|---|---|---|---|
| **busyaprime** (Akmal Xodarev) | the two exact rewrites of the DQI decoder (clearing blocks by measurement, product caching); this notebook | [dataset](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) | Apache 2.0 |
| manwithacat | the two-wire clean ladder for measurement-based uncomputation of six-input flags (Hedge Wizardry HW-07), used at 12 of our 24 sites | [notebook](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook) | Apache 2.0 |
| Ahmed Berat Ozer | the idea of product caching on the DQI line, as manwithacat's briefing of 31 October describes his 900-qubit circuit; the invsqrt phase patch | [discussion](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/754605) | idea |
| SomeJoint (Agniv Banerjee's team) | the 892-qubit decoder under our caching, as their output of 1 November 16:58 UTC ships it; their notebook credits GPT-6 Astra Pro, Claude, Ali Haydar Ozdag (flag carry), yoyobar (one-carry history) and the earlier decoder line | [notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi), [dataset](https://www.kaggle.com/datasets/somejoint/qcc-dqi-892-q26) | Apache 2.0 |
| Ehan Mallick, yoyobar, Ahmed Berat Ozer, mrmeows, busyaprime | invsqrt, unchanged: Ehan Mallick's two 3-to-2 AND merges on yoyobar's 62-qubit circuit, after Ahmed Berat Ozer's phase patch and the 63-qubit line of mrmeows and ours | as [heliosli's notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) ships it ([dataset](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live)) | Apache 2.0 |
| heliosli; Lera (marculera); Dmitrii Gluzdov; manwithacat; Ahmed Berat Ozer | chemistry, unchanged: Lera's second (9, 143, 40) factorization with its words and angle circuits k = 1 to 13, Dmitrii Gluzdov's circuits k = 0 and 14 as manwithacat assembled them, table weights by heliosli continuing Ahmed Berat Ozer's refinement (as heliosli's notebook credits them) | [heliosli's notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy), output of 2 November 00:13 UTC ([dataset](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live)) | Apache 2.0 |
| Google Quantum AI | the three verifiers and kickmix, run unchanged | [competition data](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data) | Apache 2.0 |

Version 13 had the same DQI rewrites with one caching pass and manwithacat's chemistry assembly (LB 23.90944); version 12 had the caching alone (LB 23.86954). Earlier versions (up to version 11) told the story of our invsqrt and chemistry work; they stay in
the version history.